<a href="https://colab.research.google.com/github/ALRIER/Omnichannel/blob/main/Code/GADataGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Simulations

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)
n_consumers = 300000

# === 0. Demographics ===
age = np.clip(np.random.normal(loc=35, scale=10, size=n_consumers), 18, 80)
gender = np.random.choice(["Male", "Female", "Other"], size=n_consumers, p=[0.48, 0.50, 0.02])
income_level = np.random.choice(["Low", "Medium", "High"], size=n_consumers, p=[0.3, 0.5, 0.2])

# === Derived Variable: Price Sensitivity ===
income_sensitivity = np.where(income_level == "Low", 0.3,
                      np.where(income_level == "Medium", 0.15, -0.1))

gender_sensitivity = np.where(gender == "Female", 0.1,
                       np.where(gender == "Other", 0.05, 0))

age_scaled = (age - 18) / (80 - 18)  # scale age to [0,1]
age_sensitivity = 0.25 * age_scaled

# Final sensitivity score (clipped between 0 and 1)
price_sensitivity = np.clip(
    0.4 + income_sensitivity + gender_sensitivity + age_sensitivity + np.random.normal(0, 0.05, n_consumers),
    0, 1
)

# === 1. LATENT VARIABLES ===

privacy_risk = np.random.beta(2, 5, size=n_consumers)
financial_risk = np.random.beta(2.5, 4, size=n_consumers)
product_risk = np.random.beta(3, 3, size=n_consumers)
perceived_risk = (privacy_risk + financial_risk + product_risk) / 3

# ===  Assign product category and realistic price ===

product_type = np.random.choice(["Food", "Clothing", "Tech"], size=n_consumers, p=[0.4, 0.4, 0.2])
price = np.where(
    product_type == "Food",
    np.random.lognormal(mean=np.log(400), sigma=0.2, size=n_consumers),
    np.where(
        product_type == "Clothing",
        np.random.lognormal(mean=np.log(142), sigma=0.3, size=n_consumers),
        np.random.lognormal(mean=np.log(100), sigma=0.5, size=n_consumers),
    ),
)

# Generate Perceived Value (PV) from PR and Price
alpha_0, alpha_1, alpha_2 = 0.6, -0.002, -0.9  # Adjusted alpha1 for price magnitude
epsilon_pv = np.random.normal(loc=0, scale=0.1, size=n_consumers)
perceived_value = (
    alpha_0 + alpha_1 * price + alpha_2 * perceived_risk + epsilon_pv
)
perceived_value = (perceived_value - perceived_value.min()) / (perceived_value.max() - perceived_value.min())

# === 2. PROXIES (with Correlations) ===

satisfaction = np.random.normal(loc=perceived_value * 6 + 1, scale=0.5)
satisfaction_ordinal = np.clip(np.round(satisfaction), 1, 7)
engagement = np.random.poisson(lam=perceived_value * 8 + 2 + 0.3 * satisfaction_ordinal)
engagement_ordinal = np.clip(np.round(engagement), 1, 7)
repeat_purchase = np.random.binomial(n=1, p=perceived_value)

privacy_proxy = np.clip(np.round(privacy_risk + np.random.normal(0, 0.05, n_consumers)), 1, 7)
financial_proxy = np.clip(np.round(financial_risk + np.random.normal(0, 0.05, n_consumers)), 1, 7)
product_proxy = np.clip(np.round(product_risk + np.random.normal(0, 0.05, n_consumers)), 1, 7)

# === 3. BEHAVIORAL VARIABLES ===

purchase_amount = np.random.gamma(shape=2 + perceived_value * 3 + 0.2 * engagement, scale=5)
basket_size = np.random.poisson(lam=perceived_value * 5 + 1 + 0.1 * purchase_amount / 20)
discount_sensitivity = np.random.beta(a=2, b=5, size=n_consumers)
return_rate = np.random.beta(a=(1 - perceived_value) * 5 + 1, b=perceived_value * 5 + 1)

# === Channel Effects ===

channel_used = np.random.choice(["Mobile", "Desktop", "In-Store"], size=n_consumers, p=[0.4, 0.4, 0.2])
channel_mod = np.where(channel_used == "Mobile", 0.85,
                np.where(channel_used == "In-Store", 1.20, 1.0))

# Digital Engagement
time_on_page = np.random.gamma(shape=perceived_value * 3 + 1, scale=30) * channel_mod
bounce_rate = np.clip((1 - perceived_value + np.random.normal(0, 0.05, n_consumers)) / channel_mod, 0, 1)
ctr = np.random.beta(a=perceived_value * 3 + 1, b=(1 - perceived_value) * 3 + 1)
time_to_first_action = np.random.exponential(scale=30 / (perceived_value + 0.01)) / channel_mod

# Omnichannel
switching_freq = np.random.poisson(lam=1.5 * (1 - perceived_risk))
online_offline_ratio = np.random.beta(a=perceived_value * 5 + 1, b=(1 - perceived_value) * 5 + 1)
geo_engagement = np.clip(np.random.normal(loc=0.5, scale=0.15, size=n_consumers), 0, 1)

# Affective / Loyalty
nps = np.clip(np.round(perceived_value * 100 - 50 + np.random.normal(0, 10, n_consumers)), -100, 100)
brand_advocacy = np.random.binomial(n=1, p=perceived_value)
loyalty_program_usage = np.random.poisson(lam=perceived_value * 3)
clv = np.random.lognormal(mean=3 + perceived_value * 2, sigma=0.5)

# Purchase Intention
purchase_intention = np.random.beta(a=perceived_value * 5 + 1, b=(1 - perceived_value) * 5 + 1)

# === 4. COMPILE DATAFRAME ===

df = pd.DataFrame({
    "age": age,
    "gender": gender,
    "income_level": income_level,
    "product_type": product_type,
    "price": price,
    "privacy_risk": privacy_risk,
    "financial_risk": financial_risk,
    "product_risk": product_risk,
    "perceived_risk": perceived_risk,
    "perceived_value": perceived_value,
    "satisfaction": satisfaction_ordinal,
    "engagement": engagement_ordinal,
    "repeat_purchase": repeat_purchase,
    "privacy_proxy": privacy_proxy,
    "financial_proxy": financial_proxy,
    "product_proxy": product_proxy,
    "purchase_amount": purchase_amount,
    "basket_size": basket_size,
    "discount_sensitivity": discount_sensitivity,
    "return_rate": return_rate,
    "channel_used": channel_used,
    "time_on_page": time_on_page,
    "bounce_rate": bounce_rate,
    "ctr": ctr,
    "time_to_first_action": time_to_first_action,
    "switching_freq": switching_freq,
    "online_offline_ratio": online_offline_ratio,
    "geo_engagement": geo_engagement,
    "nps": nps,
    "brand_advocacy": brand_advocacy,
    "loyalty_program_usage": loyalty_program_usage,
    "clv": clv,
    "purchase_intention": purchase_intention,
    "price_sensitivity": price_sensitivity
})


In [ ]:
df.head()

,age,gender,income_level,product_type,price,privacy_risk,financial_risk,product_risk,perceived_risk,perceived_value,...,time_to_first_action,switching_freq,online_offline_ratio,geo_engagement,nps,brand_advocacy,loyalty_program_usage,clv,purchase_intention,price_sensitivity
0,39.967142,Female,Low,Food,373.356908,0.154343,0.385503,0.497380,0.345742,0.613148,...,10.220147,1,0.437378,0.722445,5.0,0,1,82.213685,0.682548,0.813155
1,33.617357,Male,Medium,Tech,149.101198,0.432779,0.116585,0.641527,0.396964,0.711527,...,9.301239,2,0.640298,0.839156,21.0,1,6,112.635620,0.885570,0.635843
2,41.476885,Male,Low,Tech,56.982016,0.493841,0.263285,0.612841,0.456656,0.795038,...,18.034746,0,0.587914,0.523739,20.0,0,4,83.735907,0.818073,0.849002
3,50.230299,Male,Low,Food,205.814011,0.279667,0.236211,0.674472,0.396783,0.693884,...,7.986842,0,0.671446,0.763149,11.0,1,3,61.985766,0.770273,0.824241
4,32.658466,Female,Low,Clothing,148.314635,0.218054,0.201653,0.196824,0.205510,0.848358,...,11.725908,1,0.813777,0.599519,64.0,1,4,104.380100,0.845687,0.882164


#Save to CSV

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   age                    300000 non-null  float64
 1   gender                 300000 non-null  object 
 2   income_level           300000 non-null  object 
 3   product_type           300000 non-null  object 
 4   price                  300000 non-null  float64
 5   privacy_risk           300000 non-null  float64
 6   financial_risk         300000 non-null  float64
 7   product_risk           300000 non-null  float64
 8   perceived_risk         300000 non-null  float64
 9   perceived_value        300000 non-null  float64
 10  satisfaction           300000 non-null  float64
 11  engagement             300000 non-null  int64  
 12  repeat_purchase        300000 non-null  int64  
 13  privacy_proxy          300000 non-null  float64
 14  financial_proxy        300000 non-nu

In [ ]:
# 1. Select input variables (drop target)
input_vars = [c for c in df.columns if c != 'purchase_intention']
X = df[input_vars].copy()
y = df['purchase_intention'].values

# 2. Encode categorical variables (example)
# Let's assume you have 'gender', 'channel_used', 'product_type'
categorical_vars = ['gender', 'channel_used', 'product_type','income_level']
for col in categorical_vars:
    if col in X.columns:
        X[col] = X[col].astype('category').cat.codes

# 3. Check for missing data
print(X.isnull().sum().sum())
X = X.fillna(0)  # or use another imputation method if preferred

# 4. (Optional) Normalize all features
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)


0


In [ ]:
import math
num_splits = 4
chunk_size = math.ceil(len(X) / num_splits)
for i in range(num_splits):
    start = i * chunk_size
    end = (i + 1) * chunk_size
    chunk = df.iloc[start:end]
    chunk.to_csv(f"synthetic_omnichannel_dataX_part{i+1}.csv", index=False)


In [ ]:
y_df = pd.DataFrame({'purchase_intention': y})
y_df.to_csv('synthetic_omnichannel_dataY.csv', index=False)


In [ ]:
n = X_scaled.shape[0]
chunk_size = n // 4
X1 = X_scaled[:chunk_size]
X2 = X_scaled[chunk_size:2*chunk_size]
X3 = X_scaled[2*chunk_size:3*chunk_size]
X4 = X_scaled[3*chunk_size:]
df_X1 = pd.DataFrame(X1, columns=input_vars)
df_X2 = pd.DataFrame(X2, columns=input_vars)
df_X3 = pd.DataFrame(X3, columns=input_vars)
df_X4 = pd.DataFrame(X4, columns=input_vars)
df_X1.to_csv("synthetic_omnichannel_dataX_scaled_part1.csv", index=False)
df_X2.to_csv("synthetic_omnichannel_dataX_scaled_part2.csv", index=False)
df_X3.to_csv("synthetic_omnichannel_dataX_scaled_part3.csv", index=False)
df_X4.to_csv("synthetic_omnichannel_dataX_scaled_part4.csv", index=False)


#Genetic Algorithm

NOTE: 1. Targets as "Guidelines" (not hard constraints)
The fitness function will be less punitive: agents and populations will not be forced to exactly match the means/SDs/correlations, but will be rewarded for being “reasonably close.” This allows more independence and diversity in your population.

We'll also reduce the pressure of GA optimization: the population will have more variance, so you’ll see more “naturalistic” spread.



In [1]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 4.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import copy
from scipy.stats import truncnorm, beta, gamma, poisson, lognorm
from deap import base, creator, tools, algorithms

In [15]:
# -*- coding: utf-8 -*-
"""
Created on Tue Jul 22 17:23:29 2025

@author: AlvaroRivera-Eraso
"""

#%%
import numpy as np
import pandas as pd
import copy
from scipy.stats import truncnorm, beta, gamma, poisson, lognorm
from deap import base, creator, tools, algorithms
import concurrent.futures
import os


#%%
# 1. GENE NAMES AND VARIABLE TYPES (order is critical!)
GENE_NAMES = [
    "age", "gender", "income",
    "privacy_risk", "financial_risk", "product_risk", "perceived_risk",
    "perceived_value", "engagement", "satisfaction", "repeat_purchase",
    "privacy_proxy", "financial_proxy", "product_proxy",
    "price_sensitivity", "discount_sensitivity", "purchase_amount", "basket_size", "return_rate",
    "channel_used", "time_on_page", "bounce_rate", "ctr", "time_to_first_action",
    "switching_freq", "online_offline_ratio", "geo_engagement",
    "nps", "loyalty_program_usage", "clv", "brand_advocacy", "purchase_intention"
]

# 1A. GROUP DEFINITIONS
VARIABLE_GROUPS = {
    'latent': ['privacy_risk', 'financial_risk', 'product_risk', 'perceived_risk'],
    'proxies': ['satisfaction', 'engagement', 'repeat_purchase', 'privacy_proxy', 'financial_proxy', 'product_proxy'],
    'behavioral': ['purchase_amount', 'basket_size', 'discount_sensitivity', 'return_rate'],
    'channel': ['channel_used', 'time_on_page', 'bounce_rate', 'ctr', 'time_to_first_action'],
    'omnichannel': ['switching_freq', 'online_offline_ratio', 'geo_engagement'],
    'loyalty': ['nps', 'loyalty_program_usage', 'clv', 'brand_advocacy', 'purchase_intention'],
    'sensitivity': ['price_sensitivity'],
}

# 2. TARGET STATS
TARGET = {
    'privacy_risk': {'mean': 4.0, 'sd': 1.2},
    'financial_risk': {'mean': 3.8, 'sd': 1.2},
    'product_risk': {'mean': 4.1, 'sd': 1.2},
    'perceived_risk': {'mean': 4.0, 'sd': 1.2},
    'perceived_value': {'mean': 4.0, 'sd': 1.2},
    'purchase_amount': {'mean': 50, 'sd': 40},
    'basket_size': {'mean': 2.7, 'sd': 1.5},
    'discount_sensitivity': {'mean': 0.40, 'sd': 0.10},
    'return_rate': {'mean': 0.22, 'sd': 0.12},
    'time_on_page': {'mean': 175, 'sd': 120},
    'bounce_rate': {'mean': 0.47, 'sd': 0.13},
    'ctr': {'mean': 0.026, 'sd': 0.015},
    'time_to_first_action': {'mean': 7.8, 'sd': 2.0},
    'switching_freq': {'mean': 0.6, 'sd': 0.3},
    'online_offline_ratio': {'mean': 0.375, 'sd': 0.15},
    'geo_engagement': {'mean': 0.5, 'sd': 0.15},
    'nps': {'mean': 27, 'sd': 35},
    'loyalty_program_usage': {'mean': 1.8, 'sd': 1.0},
    'clv': {'mean': 800, 'sd': 2400},
    'purchase_intention': {'mean': 0.67, 'sd': 0.15},
    'price_sensitivity': {'mean': 0.33, 'sd': 0.16},
}
TARGET_CORRS = {
    ('privacy_risk','financial_risk'): 0.6,
    ('satisfaction','engagement'): 0.7,
    ('perceived_value','engagement'): 0.75,
    ('perceived_value','satisfaction'): 0.75,
    ('purchase_amount','basket_size'): 0.45,
    ('perceived_value','purchase_amount'): 0.45,
    ('discount_sensitivity','price_sensitivity'): 0.35,
}

# 3. ENCODING FOR CATEGORICALS (by index)
GENDER_MAP = ["Male", "Female", "Other"]
INCOME_MAP = ["Low", "Medium", "High"]
CHANNEL_MAP = ["Mobile", "Desktop", "In-Store"]

# 4. VARIABLE SAMPLERS
def truncnorm_var(mean, sd, low, high):
    a, b = (low-mean)/sd, (high-mean)/sd
    return truncnorm.rvs(a, b, loc=mean, scale=sd)
def likert7(mean, sd):
    val = int(round(truncnorm_var(mean, sd, 1, 7)))
    return np.clip(val, 1, 7)
def beta_var(a, b, low=0, high=1):
    return np.clip(beta.rvs(a, b), low, high)
def gamma_var(mean, sd):
    shape = (mean/sd)**2
    scale = sd**2/mean
    return gamma.rvs(shape, scale=scale)
def poisson_var(lmbd):
    return max(1, poisson.rvs(lmbd))
def lognorm_var(mean, sd):
    sigma = np.sqrt(np.log(1 + (sd/mean)**2))
    mu = np.log(mean) - 0.5*sigma**2
    return lognorm.rvs(sigma, scale=np.exp(mu))
def bounded_normal(mean, sd, low, high):
    return np.clip(np.random.normal(mean, sd), low, high)

# --- RULES (CONSTRAINTS) MODULARIZATION ---
def latent_constraints(ind_dict):
    # Privacy, financial, product risk must be Likert 1-7
    for var in ['privacy_risk', 'financial_risk', 'product_risk']:
        if not (1 <= ind_dict[var] <= 7):
            return False, f"{var} out of Likert range"
    # Perceived risk is the mean of the three
    calc = np.mean([ind_dict['privacy_risk'], ind_dict['financial_risk'], ind_dict['product_risk']])
    if not np.isclose(ind_dict['perceived_risk'], calc, atol=0.2):
        return False, "perceived_risk not matching composite"
    return True, None

def proxies_constraints(ind_dict):
    for var in ['satisfaction', 'engagement', 'privacy_proxy', 'financial_proxy', 'product_proxy']:
        if not (1 <= ind_dict[var] <= 7):
            return False, f"{var} out of Likert range"
    if not ind_dict['repeat_purchase'] in [0,1]:
        return False, "repeat_purchase not binary"
    return True, None

def behavioral_constraints(ind_dict):
    if not (ind_dict['purchase_amount'] > 0):
        return False, "purchase_amount not positive"
    if not (ind_dict['basket_size'] >= 1):
        return False, "basket_size < 1"
    if not (0 <= ind_dict['discount_sensitivity'] <= 1):
        return False, "discount_sensitivity out of bounds"
    if not (0 <= ind_dict['return_rate'] <= 1):
        return False, "return_rate out of bounds"
    return True, None

def channel_constraints(ind_dict):
    if not (ind_dict['time_on_page'] > 0):
        return False, "time_on_page not positive"
    if not (0 <= ind_dict['bounce_rate'] <= 1):
        return False, "bounce_rate out of bounds"
    if not (0 <= ind_dict['ctr'] <= 1):
        return False, "ctr out of bounds"
    if not (0 <= ind_dict['time_to_first_action'] <= 30):
        return False, "time_to_first_action out of bounds"
    return True, None

def loyalty_constraints(ind_dict):
    if not (-100 <= ind_dict['nps'] <= 100):
        return False, "nps out of bounds"
    if not (ind_dict['loyalty_program_usage'] >= 1):
        return False, "loyalty_program_usage < 1"
    if not (ind_dict['clv'] > 0):
        return False, "clv not positive"
    if not (ind_dict['brand_advocacy'] in [0,1]):
        return False, "brand_advocacy not binary"
    if not (0 <= ind_dict['purchase_intention'] <= 1):
        return False, "purchase_intention out of bounds"
    return True, None

def sensitivity_constraints(ind_dict):
    if not (0 <= ind_dict['price_sensitivity'] <= 1):
        return False, "price_sensitivity out of bounds"
    return True, None

# GROUP RULES MAP
GROUP_RULES = {
    'latent': [latent_constraints],
    'proxies': [proxies_constraints],
    'behavioral': [behavioral_constraints],
    'channel': [channel_constraints],
    'loyalty': [loyalty_constraints],
    'sensitivity': [sensitivity_constraints],
}

# Apply relevant group rules to an individual (as dict)
def check_group_rules(ind_dict):
    errors = []
    for group, variables in VARIABLE_GROUPS.items():
        if any(v in ind_dict for v in variables):
            for rule_fn in GROUP_RULES.get(group, []):
                ok, msg = rule_fn(ind_dict)
                if not ok:
                    errors.append(f"{group}: {msg}")
    return errors

# 5. INDIVIDUAL GENERATION
def make_individual_list():
    try:
        age = int(truncnorm_var(35, 10, 18, 80))
        gender = np.random.choice([0,1,2], p=[0.48, 0.5, 0.02])
        income = np.random.choice([0,1,2], p=[0.3,0.5,0.2])
        privacy_risk = likert7(TARGET['privacy_risk']['mean'], TARGET['privacy_risk']['sd'])
        financial_risk = likert7(TARGET['financial_risk']['mean'], TARGET['financial_risk']['sd'])
        product_risk = likert7(TARGET['product_risk']['mean'], TARGET['product_risk']['sd'])
        perceived_risk = np.mean([privacy_risk,financial_risk,product_risk])
        perceived_value = likert7(TARGET['perceived_value']['mean'], 1.0)
        engagement = likert7(perceived_value, 1.0)
        satisfaction = likert7(engagement, 1.0)
        repeat_purchase = int(1 if satisfaction >= 4 and engagement >= 4 else 0)
        privacy_proxy = likert7(privacy_risk, 0.7)
        financial_proxy = likert7(financial_risk, 0.7)
        product_proxy = likert7(product_risk, 0.7)
        price_sensitivity = bounded_normal(TARGET['price_sensitivity']['mean'] + 0.02*(age-35) - 0.07*income, TARGET['price_sensitivity']['sd'], 0, 1)
        discount_sensitivity = bounded_normal(TARGET['discount_sensitivity']['mean'] + 0.35*price_sensitivity, 0.08, 0, 1)
        purchase_amount = gamma_var(TARGET['purchase_amount']['mean'] + 8*(perceived_value-4), TARGET['purchase_amount']['sd'])
        basket_size = poisson_var(TARGET['basket_size']['mean'] + 0.2*(purchase_amount/50))
        return_rate = beta_var(2 + price_sensitivity*2 + perceived_risk/2, 6)
        channel_used = np.random.choice([0,1,2], p=[0.54,0.42,0.04])
        time_on_page = bounded_normal(TARGET['time_on_page']['mean'] + 10*engagement, TARGET['time_on_page']['sd'], 20, 1200)
        bounce_rate = bounded_normal(TARGET['bounce_rate']['mean'] - 0.04*engagement, TARGET['bounce_rate']['sd'], 0, 1)
        ctr = bounded_normal(TARGET['ctr']['mean'] + 0.001*engagement, TARGET['ctr']['sd'], 0, 1)
        time_to_first_action = bounded_normal(TARGET['time_to_first_action']['mean'] - 0.3*perceived_value, 2, 0, 30)
        switching_freq = poisson_var(TARGET['switching_freq']['mean'] + 0.6*perceived_risk/7 - 0.2*perceived_value/7)
        online_offline_ratio = beta_var(3 + 0.5*(age/40), 5)
        geo_engagement = np.clip(np.random.normal(0.5 + 0.07*(switching_freq-1), 0.15), 0, 1)
        nps = int(bounded_normal(27 + 9*perceived_value - 6*perceived_risk, 35, -100, 100))
        loyalty_program_usage = poisson_var(1.8 + 0.2*perceived_value + 0.1*engagement)
        clv = lognorm_var(800 + 20*loyalty_program_usage, 2400)
        brand_advocacy = int(np.random.rand() < (0.5*perceived_value/7 - 0.15*perceived_risk/7 + 0.45))
        purchase_intention = beta_var(2 + perceived_value/2, 2 + (7-perceived_value)/2)
        individual = [
            age, gender, income,
            privacy_risk, financial_risk, product_risk, perceived_risk,
            perceived_value, engagement, satisfaction, repeat_purchase,
            privacy_proxy, financial_proxy, product_proxy,
            price_sensitivity, discount_sensitivity, purchase_amount, basket_size, return_rate,
            channel_used, time_on_page, bounce_rate, ctr, time_to_first_action,
            switching_freq, online_offline_ratio, geo_engagement,
            nps, loyalty_program_usage, clv, brand_advocacy, purchase_intention
        ]
        if len(individual) != len(GENE_NAMES):
            print("DEBUG: Incorrect individual length at creation!", len(individual), individual)
        return individual
    except Exception as e:
        print(f"DEBUG: Exception in make_individual_list: {e}")
        raise

def list_to_dict(individual):
    if not isinstance(individual, (list, tuple)) or len(individual) != len(GENE_NAMES):
        print("DEBUG: Corrupt individual at list_to_dict:", individual)
        raise TypeError("Individual is not a list/tuple of correct length!")
    d = dict(zip(GENE_NAMES, individual))
    d['gender'] = GENDER_MAP[int(d['gender'])]
    d['income'] = INCOME_MAP[int(d['income'])]
    d['channel_used'] = CHANNEL_MAP[int(d['channel_used'])]
    return d

def is_valid_ind(ind):
    valid = isinstance(ind, (list, tuple)) and len(ind) == len(GENE_NAMES)
    if not valid:
        print("DEBUG: Invalid (wrong length) individual:", ind)
        return False
    ind_dict = dict(zip(GENE_NAMES, ind))
    errors = check_group_rules(ind_dict)
    if errors:
        print("DEBUG: Group rule errors:", errors, "\nIndividual:", ind_dict)
        return False
    return True

# 6. GA/DEAP SETUP
POP_SIZE = 100
NGEN = 100
N_AGENTS = 101

# Remove pre-existing classes
for name in ["FitnessMin", "Individual"]:
    if hasattr(creator, name):
        delattr(creator, name)

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)
toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, make_individual_list)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual(ind):
    if not is_valid_ind(ind):
        return (1e6,)  # Penalización máxima para inválidos
    ind_dict = list_to_dict(ind)
    score = 0
    # Evaluar targets individuales
    for v, tg in TARGET.items():
        val = ind_dict[v]
        score += (val - tg['mean'])**2/(tg['sd']**2 * 2)
    return (score,)

def evaluate_population(pop):
    df = pd.DataFrame([list_to_dict(ind) for ind in pop if is_valid_ind(ind)])
    if df.empty:
        return (1e6,)
    score = 0
    # Medias y SDs
    for v, tg in TARGET.items():
        if v in df.columns:
            m, s = df[v].mean(), df[v].std()
            score += (m-tg['mean'])**2/(tg['sd']**2*2)
            score += (s-tg['sd'])**2/(tg['sd']**2*2)
    # Correlaciones
    for (v1,v2), ctarget in TARGET_CORRS.items():
        if v1 in df.columns and v2 in df.columns:
            r = df[[v1,v2]].corr().iloc[0,1]
            score += ((r-ctarget)**2)/2
    return (score,)

# Cruzamiento, mutación, reparación en mutación
def repair_ind(ind):
    # Reparar hasta que sea válido (máximo 5 intentos)
    for _ in range(5):
        if is_valid_ind(ind):
            return ind
        # Mutar 1 gen aleatorio
        idx = np.random.randint(len(ind))
        ind[idx] = toolbox.individual()[idx]
    # Si no se puede reparar, reemplazar por nuevo válido
    new_ind = toolbox.individual()
    while not is_valid_ind(new_ind):
        new_ind = toolbox.individual()
    return new_ind

def mutate_individual(ind):
    ind = copy.deepcopy(ind)
    n_genes = np.random.randint(1, 5)  # Change up to 5 genes per mutation
    for _ in range(n_genes):
        idx = np.random.randint(len(ind))
        ind[idx] = toolbox.individual()[idx]
    ind = repair_ind(ind)
    return (ind,)

toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", mutate_individual)
toolbox.register("select", tools.selTournament, tournsize=4)
toolbox.register("evaluate", evaluate_individual)

def run_ga():
    pop = toolbox.population(n=POP_SIZE)
    # Reparar toda la población inicial
    pop = [repair_ind(ind) for ind in pop]

    hof = tools.HallOfFame(5)
    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("min", np.min)
    stats.register("avg", np.mean)
    for gen in range(NGEN):
        # Evaluar fitness
        fits = list(map(toolbox.evaluate, pop))
        for ind, fit in zip(pop, fits):
            ind.fitness.values = fit
        record = stats.compile(pop)
        print(f"Gen {gen}: min {record['min']:.4f}, avg {record['avg']:.4f}")

        # Selección de élite: mantener los mejores
        elite = tools.selBest(pop, 4)
        offspring = toolbox.select(pop, len(pop) - 4)
        offspring = algorithms.varAnd(offspring, toolbox, cxpb=0.5, mutpb=0.6)
        # Reparar y reevaluar todos los hijos
        offspring = [repair_ind(ind) for ind in offspring]
        pop = elite + offspring

    # Extraer los mejores individuos válidos para construir el DataFrame
    valid_inds = [ind for ind in pop if is_valid_ind(ind)]
    valid_inds.sort(key=lambda ind: toolbox.evaluate(ind)[0])
    best_sample = [list_to_dict(ind) for ind in valid_inds[:N_AGENTS]]
    df = pd.DataFrame(best_sample)
    return df

if __name__ == "__main__":
    TARGET_SIZE = 100000
    AGENTS_PER_RUN = 1000            # Or adjust as desired (500–5000 is reasonable)
    N_BATCHES = (TARGET_SIZE // AGENTS_PER_RUN) + 2  # A bit more to ensure enough unique

    print(f"CPU cores available: {os.cpu_count()}")

    all_dfs = []

    # --- PARALLEL EXECUTION STARTS HERE ---
    with concurrent.futures.ProcessPoolExecutor() as executor:
        # Map: run_ga (no arguments here, must be serializable!)
        futures = [executor.submit(run_ga) for _ in range(N_BATCHES)]
        for i, fut in enumerate(concurrent.futures.as_completed(futures), 1):
            df = fut.result()
            all_dfs.append(df)
            print(f"Batch {i}/{N_BATCHES} collected, total rows so far: {sum(len(x) for x in all_dfs)}")
    # --- PARALLEL EXECUTION ENDS HERE ---

    # Concatenate all batches
    big_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\nConcatenated DataFrame shape (before dedup): {big_df.shape}")

    # Remove duplicates from the FULL dataset
    big_df = big_df.drop_duplicates()
    print(f"Shape after dropping duplicates: {big_df.shape}")

    # If still too many, trim; if too few, you need more batches
    if len(big_df) > TARGET_SIZE:
        big_df = big_df.iloc[:TARGET_SIZE]
    elif len(big_df) < TARGET_SIZE:
        print(f"Warning: only {len(big_df)} unique agents generated. Consider running more batches or increasing mutation/randomness.")

    print("\nFinal DataFrame shape:", big_df.shape)
    print(big_df.head())
    big_df.to_csv("synthetic_agents_100k.csv", index=False)

    # Optional: quick checks
    print(big_df.describe())
    numeric_cols = big_df.select_dtypes(include=[np.number]).columns
    print(big_df[numeric_cols].corr())


Streaming output truncated to the last 5000 lines.
Individual: {'age': 25, 'gender': np.int64(1), 'income': np.int64(0), 'privacy_risk': np.int64(4), 'financial_risk': np.int64(4), 'product_risk': np.int64(4), 'perceived_risk': np.float64(4.333333333333333), 'perceived_value': np.int64(4), 'engagement': np.int64(2), 'satisfaction': np.int64(2), 'repeat_purchase': 0, 'privacy_proxy': np.int64(6), 'financial_proxy': np.int64(4), 'product_proxy': np.int64(5), 'price_sensitivity': np.float64(0.3531036988235969), 'discount_sensitivity': np.float64(0.3852780001305614), 'purchase_amount': np.float64(45.48290077152429), 'basket_size': 2, 'return_rate': np.float64(0.229849409365995), 'channel_used': np.int64(1), 'time_on_page': np.float64(186.86401871328968), 'bounce_rate': np.float64(0.49198288819205677), 'ctr': np.float64(0.0218552524255774), 'time_to_first_action': np.float64(6.977102165073242), 'switching_freq': 1, 'online_offline_ratio': np.float64(0.4096105396034857), 'geo_engagement': np